## Tapis Pipelines: Hello World & Sequential Tasks
#### 1. Setup and Authentication
First, we ensure the Tapis Python SDK (tapipy) is installed and initialize the client.

In [10]:
# Install tapipy if not already present
!pip install tapipy
!pip install dotenv

In [13]:
from tapipy.tapis import Tapis
from dotenv import load_dotenv
import json
import urllib3
import base64
import os

# Load environment variables
load_dotenv()

# Extraer variables de entorno
BASE_URL = os.getenv('TAPIS_BASE_URL')
USERNAME = os.getenv('TAPIS_USERNAME')
PASSWORD = os.getenv('TAPIS_PASSWORD')
GROUP_ID = os.getenv('TAPIS_GROUP_ID')

# Network Configuration
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize the client
t = Tapis(
    base_url=BASE_URL, 
    username=USERNAME, 
    password=PASSWORD, 
    verify=False
)
t.get_tokens()

print("✅ Successfully authenticated with Tapis.")

✅ Successfully authenticated with Tapis.


#### 2. Registering a "Hello World" Task
In Tapis, a Task is the atomic unit of work. Below is the simplest form using a standard Docker image to run a "Hello World" command.

+ Example Task Definition:

+ ID: hello-world-task

+ Type: image (runs a container)

+ Runtime: docker

+ Command: The specific script or command to execute.

In [14]:
os.makedirs('pipelines', exist_ok=True)

def to_tapis_base64(code_str):
    # Codificamos y aseguramos que el padding sea correcto
    return base64.b64encode(code_str.encode('utf-8')).decode('utf-8')

code_1 = to_tapis_base64("print('Task 1 completed successfully')\n")
code_2 = to_tapis_base64("print('Task 2 completed successfully')\n")

pipeline_content = {
    "id": "hello-world-pipeline",
    "description": "Pipeline con Base64 manual para evitar error de padding",
    "type": "workflow",
    "tasks": [
        {
            "id": "task-one",
            "type": "function",
            "runtime": "python:3.11-slim",
            "installer": "pip",
            "code": code_1 
        },
        {
            "id": "task-two",
            "type": "function",
            "runtime": "python:3.11-slim",
            "installer": "pip",
            "code": code_2,
            "depends_on": [{"id": "task-one"}]
        }
    ]
}

with open('pipelines/hello.json', 'w') as f:
    json.dump(pipeline_content, f, indent=2)

print("✅ Document updated with the code pre-coded in Base64.")

✅ Document updated with the code pre-coded in Base64.


#### 3. Creating a Pipeline with Two Sequential Tasks
To run tasks "back-to-back," we define a Pipeline. We use the depends_on field to create a dependency, ensuring Task B only starts after Task A completes successfully.

In [15]:
try:
    with open('pipelines/hello.json', 'r') as f:
        pipeline_data = json.load(f)
    p_id = pipeline_data['id']

    # Forzar eliminación para limpiar cualquier residuo del error de padding
    try:
        t.workflows.deletePipeline(group_id=GROUP_ID, pipeline_id=p_id)
    except: pass
    
    t.workflows.createPipeline(group_id=GROUP_ID, **pipeline_data)
    print(f"✅ Pipeline '{p_id}' re-registrado.")

    run = t.workflows.runPipeline(group_id=GROUP_ID, pipeline_id=p_id)
    last_run_uuid = run.uuid
    print(f"\n🚀 New Execution Started: {last_run_uuid}")

except Exception as e:
    print(f"❌ Error: {e}")

✅ Pipeline 'hello-world-pipeline' re-registrado.

🚀 NUEVA EJECUCIÓN INICIADA: 2ee26929-102e-4be3-9f3d-400966646159


#### 4. Monitoring the Pipeline
Once registered, you can trigger a "run" of the pipeline and monitor its status.


In [17]:
import time

# --- PEGA EL UUID AQUÍ ---
target_run_uuid = last_run_uuid 
# -------------------------

if 'last_run_uuid' in locals():
    print(f"Monitoreando Run: {last_run_uuid}")
    while True:
        res = t.workflows.getPipelineRun(group_id="test_s_sterling", pipeline_id="hello-world-pipeline", pipeline_run_uuid=last_run_uuid)
        print(f"[{time.strftime('%H:%M:%S')}] Estado: {res.status}")
        if res.status in ['completed', 'failed', 'cancelled']:
            if res.status == 'failed':
                # Intentamos obtener el mensaje de error detallado si está disponible
                print(f"Detalles del fallo: {getattr(res, 'message', 'No hay mensaje detallado')}")
            break
        time.sleep(5)
else:
    print("No hay UUID en memoria.")

Monitoreando Run: 2ee26929-102e-4be3-9f3d-400966646159
[19:38:37] Estado: completed


---
### Delete Pipeline

In [33]:
group_id = "test_s_sterling"
pipeline_id = "hello-world-pipeline"

# 1. Intentar borrar el anterior para evitar conflictos de serialización
try:
    t.workflows.deletePipeline(group_id=group_id, pipeline_id=pipeline_id)
    print("Pipeline anterior eliminado para refrescar el esquema.")
except:
    pass

# 2. Registro con el nuevo tipo 'tapis_job'
try:
    t.workflows.createPipeline(
        group_id=group_id,
        **pipeline_data
    )
    print("✅ Pipeline registrado con tipo 'tapis_job'.")
    
    # 3. Intentar correrlo inmediatamente
    run = t.workflows.runPipeline(group_id=group_id, pipeline_id=pipeline_id)
    print(f"🚀 ¡Ejecución iniciada! UUID: {run.uuid}")
    
except Exception as e:
    print(f"❌ Falló nuevamente: {e}")

❌ Falló nuevamente: message: Unable to make request to Tapis server. Exception: HTTPSConnectionPool(host='portals.tapis.io', port=443): Max retries exceeded with url: /v3/workflows/groups/test_s_sterling/pipelines (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)')))
